In [1]:
# 0. Install Required Dependencies (Run this first!)\n!pip install -q torch torchvision optuna scikit-learn seaborn matplotlib pandas numpy opencv-python Pillow scikit-image scipy

## Phase 1 & 2: Dataset Loading and Exploratory Data Analysis (EDA)

#  CottonGuard AI — Exploratory Data Analysis
> Disease + Severity Classification for Cotton Leaves/Plants

**Classes (10 total):**
- Alternaria Leaf
- Bacterial Blight (Mild / Moderate / Critical)
- Curl Virus (Mild / Moderate / Critical)
- Fussarium Wilt (Mild / Moderate / Critical)

---
##  STEP 0 — Setup: Seeds, Imports, Output Directory

In [2]:
from IPython.display import display
# == SEED FIRST — before any other import ======================================
import random
import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

try:
    import torch
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    print(f"PyTorch {torch.__version__} | CUDA available: {torch.cuda.is_available()}")
except ImportError:
    torch = None
    print("PyTorch not available — skipping torch seed.")

# == Standard imports ==========================================================
import os
import json
import hashlib
import warnings
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import cv2
from PIL import Image, UnidentifiedImageError

from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.utils.class_weight import compute_class_weight

from skimage.feature import hog, graycomatrix, graycoprops
from scipy.stats import f_oneway

try:
    import torchvision.transforms as T
    from torchvision.transforms import functional as TF
    print(f"torchvision available.")
except ImportError:
    T = None
    print("torchvision not available — Step 10 will be skipped.")

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')

# == Output directory ==========================================================
OUTPUT_DIR = Path('outputs')
OUTPUT_DIR.mkdir(exist_ok=True)
print(f"Output directory: {OUTPUT_DIR.resolve()}")

# == Library versions ==========================================================
import sklearn, skimage, scipy
print(f"\n--- Library Versions ---")
print(f"numpy        : {np.__version__}")
print(f"pandas       : {pd.__version__}")
print(f"scikit-learn : {sklearn.__version__}")
print(f"scikit-image : {skimage.__version__}")
print(f"scipy        : {scipy.__version__}")
print(f"opencv       : {cv2.__version__}")
print(f"Pillow       : {Image.__version__}")

PyTorch 2.12.0+cpu | CUDA available: False


torchvision available.
Output directory: C:\Users\GNG\Documents\Shamail\DL Lab\Project\DL AGRICULTURE PROJECT\DL AGRICULTURE\outputs

--- Library Versions ---
numpy        : 2.4.4
pandas       : 3.0.2
scikit-learn : 1.8.0
scikit-image : 0.26.0
scipy        : 1.17.1
opencv       : 4.13.0
Pillow       : 12.2.0


---
##  STEP 1 — Dataset Inventory

In [3]:
# == Dataset folder mapping ====================================================
# Each entry: (folder_name, label, has_severity_subfolders)
DATASET_ROOT = Path('.')
IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.tiff', '.tif', '.webp'}

FOLDER_CONFIG = [
    # (folder,              base_label,         has_severity)
    ('aug_Alternaria_Leaf', 'Alternaria Leaf',   False),
    ('bacterial blight',    'Bacterial Blight',  True),
    ('curl virus',          'Curl Virus',        True),
    ('fussarium wilt',      'Fussarium Wilt',    True),
]
SEVERITY_LEVELS = ['mild', 'moderate', 'critical']

def md5_hash(filepath):
    h = hashlib.md5()
    with open(filepath, 'rb') as f:
        for chunk in iter(lambda: f.read(8192), b''):
            h.update(chunk)
    return h.hexdigest()

def is_valid_image(filepath):
    try:
        with Image.open(filepath) as img:
            img.verify()
        return True
    except Exception:
        return False

records = []
corrupt_files = []
non_image_files = []

for folder_name, base_label, has_severity in FOLDER_CONFIG:
    folder_path = DATASET_ROOT / folder_name
    if not folder_path.exists():
        print(f"  [WARNING] Folder not found: {folder_path}")
        continue

    if not has_severity:
        # Alternaria: all images in root folder are REAL
        for fpath in folder_path.iterdir():
            if fpath.is_file():
                if fpath.suffix.lower() not in IMAGE_EXTS:
                    non_image_files.append(str(fpath))
                    continue
                valid = is_valid_image(fpath)
                if not valid:
                    corrupt_files.append(str(fpath))
                    continue
                records.append({
                    'path': str(fpath),
                    'label': base_label,
                    'is_augmented': False,
                    'source_type': 'raw'
                })
    else:
        # Hierarchical: folder/severity/images + folder/severity/augment_result/images
        for severity in SEVERITY_LEVELS:
            sev_path = folder_path / severity
            if not sev_path.exists():
                # try capitalised
                sev_path = folder_path / severity.capitalize()
            if not sev_path.exists():
                print(f"  [WARNING] Severity folder not found: {folder_path}/{severity}")
                continue

            label = f"{base_label} - {severity.capitalize()}"

            # Real images directly in severity folder
            for fpath in sev_path.iterdir():
                if fpath.is_dir():
                    continue  # skip subdirs (augment_result handled below)
                if fpath.suffix.lower() not in IMAGE_EXTS:
                    non_image_files.append(str(fpath))
                    continue
                valid = is_valid_image(fpath)
                if not valid:
                    corrupt_files.append(str(fpath))
                    continue
                records.append({
                    'path': str(fpath),
                    'label': label,
                    'is_augmented': False,
                    'source_type': 'raw'
                })

            # Augmented images in augment_result subfolder
            aug_path = sev_path / 'augment_result'
            if aug_path.exists():
                for fpath in aug_path.rglob('*'):
                    if not fpath.is_file():
                        continue
                    if fpath.suffix.lower() not in IMAGE_EXTS:
                        non_image_files.append(str(fpath))
                        continue
                    valid = is_valid_image(fpath)
                    if not valid:
                        corrupt_files.append(str(fpath))
                        continue
                    records.append({
                        'path': str(fpath),
                        'label': label,
                        'is_augmented': True,
                        'source_type': 'augmented'
                    })

df_all = pd.DataFrame(records)
print(f"Total images found   : {len(df_all)}")
print(f"  Raw images         : {(~df_all['is_augmented']).sum()}")
print(f"  Augmented images   : {df_all['is_augmented'].sum()}")
print(f"Corrupt files        : {len(corrupt_files)}")
print(f"Non-image files      : {len(non_image_files)}")

# == Duplicate detection via MD5 ===============================================
print("\nComputing MD5 hashes for duplicate detection...")
df_all['md5'] = df_all['path'].apply(md5_hash)
dup_mask = df_all.duplicated(subset='md5', keep=False)
print(f"Duplicate images (same MD5): {dup_mask.sum()}")

# == Inventory summary table ===================================================
summary = df_all.groupby(['label', 'source_type']).size().unstack(fill_value=0).reset_index()
summary.columns.name = None
for col in ['raw', 'augmented']:
    if col not in summary.columns:
        summary[col] = 0
summary['total'] = summary['raw'] + summary['augmented']
summary['%_of_total'] = (summary['total'] / summary['total'].sum() * 100).round(2)
summary = summary.rename(columns={'raw': 'raw_count', 'augmented': 'augmented_count'})
summary = summary[['label', 'raw_count', 'augmented_count', 'total', '%_of_total']]
print("\n--- Dataset Inventory ---")
display(summary)

Total images found   : 3466
  Raw images         : 1178
  Augmented images   : 2288
Corrupt files        : 0
Non-image files      : 0

Computing MD5 hashes for duplicate detection...


Duplicate images (same MD5): 833

--- Dataset Inventory ---


,label,raw_count,augmented_count,total,%_of_total
0,Alternaria Leaf,987,0,987,28.48
1,Bacterial Blight - Critical,19,300,319,9.20
2,Bacterial Blight - Mild,50,300,350,10.10
3,Bacterial Blight - Moderate,50,300,350,10.10
4,Curl Virus - Critical,11,100,111,3.20
5,Curl Virus - Mild,9,300,309,8.92
6,Curl Virus - Moderate,5,300,305,8.80
7,Fussarium Wilt - Critical,11,88,99,2.86
8,Fussarium Wilt - Mild,19,300,319,9.20
9,Fussarium Wilt - Moderate,17,300,317,9.15


---
##  STEP 2 — Stratified Split (Data-Leakage-Safe)

In [4]:
# == Use ONLY raw images for splitting =========================================
df_raw = df_all[~df_all['is_augmented']].reset_index(drop=True)
df_aug = df_all[df_all['is_augmented']].reset_index(drop=True)

X_raw = df_raw['path'].values
y_raw = df_raw['label'].values

# First split: 70% train_real, 30% temp
sss1 = StratifiedShuffleSplit(n_splits=1, test_size=0.30, random_state=SEED)
train_idx, temp_idx = next(sss1.split(X_raw, y_raw))

X_temp, y_temp = X_raw[temp_idx], y_raw[temp_idx]

# ==  FIX: remove classes with <2 samples BEFORE stratified split ===========
from collections import Counter
import numpy as np

counts_temp = Counter(y_temp)

valid_mask = np.array([counts_temp[label] >= 2 for label in y_temp])

X_temp = X_temp[valid_mask]
y_temp = y_temp[valid_mask]

# Second split: temp → 50% val, 50% test
sss2 = StratifiedShuffleSplit(n_splits=1, test_size=0.50, random_state=SEED)
val_idx_local, test_idx_local = next(sss2.split(X_temp, y_temp))

train_paths_real = set(X_raw[train_idx])
val_paths        = set(X_temp[val_idx_local])
test_paths       = set(X_temp[test_idx_local])

# == Assertions: no overlap ====================================================
assert len(train_paths_real & val_paths)  == 0, "LEAK: train ∩ val"
assert len(train_paths_real & test_paths) == 0, "LEAK: train ∩ test"
assert len(val_paths & test_paths)        == 0, "LEAK: val ∩ test"
print(" No overlap between splits.")

# == Add ALL augmented images to training AFTER split =========================
df_train_real = df_raw[df_raw['path'].isin(train_paths_real)].copy()
df_train_real['split'] = 'train'

df_aug_copy = df_aug.copy()
df_aug_copy['split'] = 'train'

train_df = pd.concat([df_train_real, df_aug_copy], ignore_index=True)

val_df  = df_raw[df_raw['path'].isin(val_paths)].copy()
val_df['split'] = 'val'

test_df = df_raw[df_raw['path'].isin(test_paths)].copy()
test_df['split'] = 'test'

# == Save split ID files =======================================================
def save_split_txt(df, fname):
    with open(OUTPUT_DIR / fname, 'w') as f:
        for _, row in df.iterrows():
            f.write(f"{row['path']}\t{row['label']}\n")

save_split_txt(train_df, 'train_ids.txt')
save_split_txt(val_df,   'val_ids.txt')
save_split_txt(test_df,  'test_ids.txt')

print(f"\n--- Split Summary ---")
print(f"Train (real)     : {len(df_train_real):>6}")
print(f"Train (augmented): {len(df_aug_copy):>6}")
print(f"Train (total)    : {len(train_df):>6}")
print(f"Val              : {len(val_df):>6}")
print(f"Test             : {len(test_df):>6}")
print(f"\nSplit files saved to: {OUTPUT_DIR.resolve()}")

# == Remove val/test paths from memory (safety) ================================
del val_paths, test_paths, X_temp, y_temp
del temp_idx, val_idx_local, test_idx_local
print("\n Val/test path sets deleted from memory.")

✅ No overlap between splits.



--- Split Summary ---
Train (real)     :    824
Train (augmented):   2288
Train (total)    :   3112
Val              :    176
Test             :    177

Split files saved to: C:\Users\GNG\Documents\Shamail\DL Lab\Project\DL AGRICULTURE PROJECT\DL AGRICULTURE\outputs

✅ Val/test path sets deleted from memory.


---
##  STEP 3 — Class Distribution (Train Only)

In [5]:
# DATA SOURCE: training split only.

label_counts = train_df['label'].value_counts().sort_values(ascending=False)
total_train  = len(train_df)

fig, ax = plt.subplots(figsize=(14, 6))
bars = sns.countplot(
    data=train_df,
    y='label',
    order=label_counts.index,
    palette='viridis',
    ax=ax
)
ax.set_title('Class Distribution — Training Set', fontsize=16, fontweight='bold')
ax.set_xlabel('Count', fontsize=13)
ax.set_ylabel('Class', fontsize=13)

for bar, (label, cnt) in zip(ax.patches, label_counts.items()):
    pct = cnt / total_train * 100
    ax.text(
        bar.get_width() + 5, bar.get_y() + bar.get_height() / 2,
        f"{cnt}  ({pct:.1f}%)", va='center', fontsize=10
    )

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

# == Imbalance ratio ===========================================================
imbalance_ratio = label_counts.max() / label_counts.min()
print(f"\nImbalance ratio (max/min): {imbalance_ratio:.2f}")
print(f"Most frequent class : {label_counts.idxmax()} ({label_counts.max()})")
print(f"Least frequent class: {label_counts.idxmin()} ({label_counts.min()})")

# == Class weights for training ================================================
classes   = np.unique(train_df['label'])
cw        = compute_class_weight('balanced', classes=classes, y=train_df['label'].values)
cw_dict   = dict(zip(classes, cw.round(4)))
print("\nClass weights (balanced):")
for k, v in cw_dict.items():
    print(f"  {k:<35}: {v}")


Imbalance ratio (max/min): 7.19
Most frequent class : Alternaria Leaf (690)
Least frequent class: Fussarium Wilt - Critical (96)

Class weights (balanced):
  Alternaria Leaf                    : 0.451
  Bacterial Blight - Critical        : 0.9942
  Bacterial Blight - Mild            : 0.929
  Bacterial Blight - Moderate        : 0.929
  Curl Virus - Critical              : 2.8815
  Curl Virus - Mild                  : 1.017
  Curl Virus - Moderate              : 1.0237
  Fussarium Wilt - Critical          : 3.2417
  Fussarium Wilt - Mild              : 0.9942
  Fussarium Wilt - Moderate          : 0.9974


---
##  STEP 4 — Image Dimension Audit (Train Only)

In [6]:
# DATA SOURCE: training split only.

dim_records = []

for _, row in train_df.iterrows():
    try:
        with Image.open(row['path']) as img:
            w, h  = img.size
            mode  = img.mode
        fsize_kb = os.path.getsize(row['path']) / 1024
        dim_records.append({
            'path': row['path'],
            'label': row['label'],
            'width': w,
            'height': h,
            'aspect_ratio': w / h if h > 0 else 0,
            'file_size_kb': fsize_kb,
            'color_mode': mode
        })
    except Exception:
        pass

df_dim = pd.DataFrame(dim_records)

# == Flagged images ============================================================
flag_ar = df_dim[(df_dim['aspect_ratio'] < 0.5) | (df_dim['aspect_ratio'] > 2)]
flag_sz = df_dim[(df_dim['width'] < 100) | (df_dim['height'] < 100)]
print(f"Flagged — unusual aspect ratio (<0.5 or >2): {len(flag_ar)}")
print(f"Flagged — min dimension < 100px           : {len(flag_sz)}")

# == Plots =====================================================================
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle('Image Dimension Audit — Training Set', fontsize=15, fontweight='bold')

# Scatter: width vs height
scatter_data = df_dim.copy()
scatter_data['class_group'] = scatter_data['label'].str.split(' - ').str[0]
sns.scatterplot(
    data=scatter_data, x='width', y='height',
    hue='class_group', alpha=0.5, ax=axes[0]
)
axes[0].set_title('Width vs Height')
axes[0].set_xlabel('Width (px)')
axes[0].set_ylabel('Height (px)')

# Histogram: file size
axes[1].hist(df_dim['file_size_kb'], bins=40, color='steelblue', edgecolor='white')
axes[1].set_title('File Size Distribution')
axes[1].set_xlabel('File Size (KB)')
axes[1].set_ylabel('Count')

# Boxplot: aspect ratio per class
short_labels = df_dim['label'].apply(lambda x: x if ' - ' not in x else x.split(' - ')[0][0] + '.' + x.split(' - ')[1][:3])
df_dim_plot = df_dim.copy()
df_dim_plot['short_label'] = short_labels
sns.boxplot(data=df_dim_plot, x='short_label', y='aspect_ratio', ax=axes[2], palette='Set2')
axes[2].set_title('Aspect Ratio per Class')
axes[2].set_xlabel('Class')
axes[2].set_ylabel('Aspect Ratio')
axes[2].tick_params(axis='x', rotation=45)
axes[2].axhline(0.5, color='red', linestyle='--', alpha=0.5, label='Threshold 0.5')
axes[2].axhline(2.0, color='red', linestyle='--', alpha=0.5, label='Threshold 2.0')
axes[2].legend(fontsize=8)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'dimension_audit.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nDimension stats (training):")
print(df_dim[['width','height','aspect_ratio','file_size_kb']].describe().round(2))

Flagged — unusual aspect ratio (<0.5 or >2): 0
Flagged — min dimension < 100px           : 0



Dimension stats (training):
         width   height  aspect_ratio  file_size_kb
count  3112.00  3112.00       3112.00       3112.00
mean   1365.92  1741.88          0.84        352.97
std     693.37   929.07          0.15        249.94
min     159.00   164.00          0.68         13.02
25%     400.00   400.00          0.75         75.24
50%    1768.00  2358.00          0.75        328.65
75%    1768.00  2358.00          1.00        542.56
max    2358.00  2358.00          1.43       1083.65


---
##  STEP 5 — Pixel Statistics & Normalization (Train Only)

In [7]:
# DATA SOURCE: training split only.

TARGET_SIZE = (640, 640)

rgb_means, rgb_stds   = [], []
brightness_list, contrast_list, label_list = [], [], []

# Sample up to 500 images per class for speed
SAMPLE_PER_CLASS = 500
sample_df = (
    train_df
    .sample(frac=1, random_state=SEED)
    .groupby('label')
    .head(SAMPLE_PER_CLASS)
    .reset_index(drop=True)
)

for _, row in sample_df.iterrows():
    try:
        img = Image.open(row['path']).convert('RGB').resize(TARGET_SIZE)
        arr = np.array(img, dtype=np.float32) / 255.0

        r_m = arr.mean(axis=(0, 1))
        r_s = arr.std(axis=(0, 1))

        gray = np.mean(arr, axis=2)
        b_m = gray.mean()
        b_s = gray.std()
        lbl = row['label']
        rgb_means.append(r_m)
        rgb_stds.append(r_s)
        brightness_list.append(b_m)
        contrast_list.append(b_s)
        label_list.append(lbl)
    except Exception as e:
        print(f"Error processing {row['path']}: {e}")
        pass

rgb_means = np.array(rgb_means)
rgb_stds  = np.array(rgb_stds)

global_mean = rgb_means.mean(axis=0).tolist() if len(rgb_means) > 0 else [0.0, 0.0, 0.0]
global_std = rgb_stds.mean(axis=0).tolist() if len(rgb_stds) > 0 else [1.0, 1.0, 1.0]

norm_stats = {
    'mean_rgb': [round(v, 6) for v in global_mean],
    'std_rgb' : [round(v, 6) for v in global_std],
    'computed_from': 'training_split_only',
    'target_size': list(TARGET_SIZE),
    'n_images_sampled': len(rgb_means)
}

with open(OUTPUT_DIR / 'norm_stats.json', 'w') as f:
    json.dump(norm_stats, f, indent=2)

print("Normalization Stats (training split only):")
print(f"  Mean RGB : {[round(v, 4) for v in global_mean]}")
print(f"  Std  RGB : {[round(v, 4) for v in global_std]}")
print(f"  Saved to : {OUTPUT_DIR / 'norm_stats.json'}")

# == Brightness & Contrast boxplots ============================================
df_bc = pd.DataFrame({'label': label_list, 'brightness': brightness_list, 'contrast': contrast_list})
df_bc['short'] = df_bc['label'].apply(lambda x: x if ' - ' not in x else x.split(' - ')[0][0]+'.'+x.split(' - ')[1][:3])

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Brightness & Contrast per Class — Training Set', fontsize=14, fontweight='bold')

sns.boxplot(data=df_bc, x='short', y='brightness', ax=axes[0], palette='Blues')
axes[0].set_title('Brightness (grayscale mean)')
axes[0].set_xlabel('Class'); axes[0].set_ylabel('Brightness')
axes[0].tick_params(axis='x', rotation=45)

sns.boxplot(data=df_bc, x='short', y='contrast', ax=axes[1], palette='Oranges')
axes[1].set_title('Contrast (grayscale std)')
axes[1].set_xlabel('Class'); axes[1].set_ylabel('Contrast')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'brightness_contrast.png', dpi=150, bbox_inches='tight')
plt.show()

Normalization Stats (training split only):
  Mean RGB : [0.5192, 0.5727, 0.4891]
  Std  RGB : [0.2571, 0.2411, 0.3069]
  Saved to : outputs\norm_stats.json


---
##  STEP 6 — Sample Image Grids (Train Only, 4×5 per class)

In [8]:
# DATA SOURCE: training split only.

GRID_ROWS, GRID_COLS = 4, 5
N_SAMPLES = GRID_ROWS * GRID_COLS

for label in sorted(train_df['label'].unique()):
    class_df = train_df[train_df['label'] == label]
    samples  = class_df.sample(min(N_SAMPLES, len(class_df)), random_state=SEED)

    fig, axes = plt.subplots(GRID_ROWS, GRID_COLS, figsize=(14, 11))
    fig.suptitle(f'Sample Grid — {label}', fontsize=14, fontweight='bold')

    for idx, (ax, (_, row)) in enumerate(zip(axes.flat, samples.iterrows())):
        try:
            img = Image.open(row['path']).convert('RGB')
            ax.imshow(img)
            ax.set_title('aug' if row['is_augmented'] else 'real', fontsize=7)
        except Exception:
            ax.text(0.5, 0.5, 'Error', ha='center', va='center')
        ax.axis('off')

    # hide unused axes
    for ax in axes.flat[len(samples):]:
        ax.axis('off')

    safe_name = label.replace(' ', '_').replace('/', '_').replace('-', '')
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f'grid_{safe_name}.png', dpi=120, bbox_inches='tight')
    plt.show()
    plt.close()

---
##  STEP 7 — HSV Color Analysis (Train Only)

In [9]:
# DATA SOURCE: training split only.

HSV_SAMPLE = 100

labels_ordered = sorted(train_df['label'].unique())
n_classes = len(labels_ordered)

fig, axes = plt.subplots(n_classes, 3, figsize=(16, n_classes * 3))
fig.suptitle('HSV Channel KDE per Class — Training Set', fontsize=14, fontweight='bold')

channel_names = ['Hue', 'Saturation', 'Value']
channel_colors = ['purple', 'green', 'orange']

for row_idx, label in enumerate(labels_ordered):
    cls_df  = train_df[train_df['label'] == label]
    samples = cls_df.sample(min(HSV_SAMPLE, len(cls_df)), random_state=SEED)

    h_vals, s_vals, v_vals = [], [], []
    for _, r in samples.iterrows():
        try:
            img_bgr = cv2.imread(r['path'])
            if img_bgr is None:
                continue
            img_bgr = cv2.resize(img_bgr, (32, 32))
            hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
            h_vals.extend(hsv[:,:,0].flatten().tolist())
            s_vals.extend(hsv[:,:,1].flatten().tolist())
            v_vals.extend(hsv[:,:,2].flatten().tolist())
        except Exception:
            pass

    for col_idx, (vals, ch_name, color) in enumerate(zip([h_vals, s_vals, v_vals], channel_names, channel_colors)):
        ax = axes[row_idx, col_idx] if n_classes > 1 else axes[col_idx]
        if vals:
            sns.kdeplot(vals, ax=ax, color=color, fill=True, alpha=0.4)
        ax.set_title(f"{label}\n{ch_name}", fontsize=8)
        ax.set_xlabel(ch_name, fontsize=7)
        ax.set_ylabel('Density', fontsize=7)
        ax.tick_params(labelsize=7)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'hsv_kde.png', dpi=120, bbox_inches='tight')
plt.show()

---
##  STEP 8 — Texture Analysis: HOG + GLCM (Train Only)

In [10]:
# DATA SOURCE: training split only.
# == HOG Visualization: 5 images per class ====================================

from skimage.feature import hog
from skimage import exposure

HOG_SAMPLE = 5

for label in labels_ordered:
    cls_df  = train_df[train_df['label'] == label]
    samples = cls_df.sample(min(HOG_SAMPLE, len(cls_df)), random_state=SEED)

    fig, axes = plt.subplots(2, HOG_SAMPLE, figsize=(15, 6))
    fig.suptitle(f'HOG Features — {label}', fontsize=12, fontweight='bold')

    for col, (_, row) in enumerate(samples.iterrows()):
        try:
            img = Image.open(row['path']).convert('L').resize((128, 128))
            arr = np.array(img)
            fd, hog_img = hog(
                arr, orientations=9,
                pixels_per_cell=(8, 8),
                cells_per_block=(2, 2),
                visualize=True
            )
            hog_img_rescaled = exposure.rescale_intensity(hog_img, in_range=(0, 10))

            axes[0, col].imshow(arr, cmap='gray')
            axes[0, col].set_title('Original', fontsize=8)
            axes[0, col].axis('off')

            axes[1, col].imshow(hog_img_rescaled, cmap='magma')
            axes[1, col].set_title('HOG', fontsize=8)
            axes[1, col].axis('off')
        except Exception as e:
            for r in range(2):
                axes[r, col].axis('off')

    safe = label.replace(' ', '_').replace('-','').replace('/','_')
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f'hog_{safe}.png', dpi=120, bbox_inches='tight')
    plt.show()
    plt.close()

In [11]:
# DATA SOURCE: training split only.
# == GLCM Analysis: 50 images per class =======================================

GLCM_SAMPLE = 50
glcm_records = []

distances  = [1]
angles     = [0, np.pi/4, np.pi/2, 3*np.pi/4]
props      = ['contrast', 'correlation', 'energy', 'homogeneity']

for label in labels_ordered:
    cls_df  = train_df[train_df['label'] == label]
    samples = cls_df.sample(min(GLCM_SAMPLE, len(cls_df)), random_state=SEED)

    for _, row in samples.iterrows():
        try:
            img = Image.open(row['path']).convert('L').resize((128, 128))
            arr = (np.array(img) // 16).astype(np.uint8)  # quantize to 16 levels
            glcm = graycomatrix(arr, distances=distances, angles=angles, levels=16, symmetric=True, normed=True)
            rec = {'label': label}
            for p in props:
                rec[p] = float(graycoprops(glcm, p).mean())
            glcm_records.append(rec)
        except Exception:
            pass

df_glcm = pd.DataFrame(glcm_records)

# == ANOVA per GLCM property ===================================================
anova_results = {}
for p in props:
    groups = [group[p].values for _, group in df_glcm.groupby('label')]
    f_stat, p_val = f_oneway(*groups)
    anova_results[p] = {'F_statistic': round(float(f_stat), 4), 'p_value': float(p_val)}
    print(f"ANOVA [{p:15s}]: F={f_stat:.4f}, p={p_val:.4e}")

# Save GLCM stats
df_glcm.to_csv(OUTPUT_DIR / 'glcm_stats.csv', index=False)
with open(OUTPUT_DIR / 'glcm_anova.json', 'w') as f:
    json.dump(anova_results, f, indent=2)
print(f"\nGLCM stats saved to: {OUTPUT_DIR / 'glcm_stats.csv'}")

# == Boxplots ==================================================================
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle('GLCM Texture Properties per Class — Training Set', fontsize=14, fontweight='bold')

df_glcm['short'] = df_glcm['label'].apply(
    lambda x: x if ' - ' not in x else x.split(' - ')[0][0]+'.'+x.split(' - ')[1][:3]
)

for ax, p in zip(axes.flat, props):
    sns.boxplot(data=df_glcm, x='short', y=p, ax=ax, palette='coolwarm')
    pval = anova_results[p]['p_value']
    ax.set_title(f"{p.capitalize()} (ANOVA p={pval:.3e})", fontsize=11)
    ax.set_xlabel('Class'); ax.set_ylabel(p.capitalize())
    ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'glcm_boxplots.png', dpi=150, bbox_inches='tight')
plt.show()

ANOVA [contrast       ]: F=89.6262, p=1.1226e-97
ANOVA [correlation    ]: F=21.8337, p=3.5072e-31
ANOVA [energy         ]: F=7.4844, p=2.7475e-10
ANOVA [homogeneity    ]: F=56.1685, p=7.2847e-70

GLCM stats saved to: outputs\glcm_stats.csv


---
##  STEP 9 — CLAHE Validation (Train Only)

In [12]:
# DATA SOURCE: training split only.

# == Identify darkest 10% images by brightness =================================
dark_threshold = df_bc['brightness'].quantile(0.10)
dark_labels_df  = df_bc[df_bc['brightness'] <= dark_threshold]

# Merge back with train_df to get paths
df_brightness_path = df_bc.copy()
df_brightness_path['path'] = sample_df['path'].values[:len(df_bc)]
dark_paths_df = df_brightness_path[df_brightness_path['brightness'] <= dark_threshold].head(8)

print(f"Brightness 10th percentile threshold: {dark_threshold:.4f}")
print(f"Darkest images selected for CLAHE demo: {len(dark_paths_df)}")

def apply_clahe(img_bgr, clip_limit=2.0, tile_grid=(8, 8)):
    lab = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_grid)
    l_eq  = clahe.apply(l)
    lab_eq = cv2.merge((l_eq, a, b))
    return cv2.cvtColor(lab_eq, cv2.COLOR_LAB2BGR)

def rms_contrast(img_gray):
    mu = img_gray.mean()
    return np.sqrt(((img_gray.astype(float) - mu) ** 2).mean())

rms_improvements = []
n_show = min(4, len(dark_paths_df))

fig, axes = plt.subplots(n_show, 2, figsize=(10, n_show * 3))
fig.suptitle('CLAHE Before/After — Darkest 10% Images', fontsize=13, fontweight='bold')

for i, (_, row) in enumerate(dark_paths_df.head(n_show).iterrows()):
    try:
        img_bgr   = cv2.imread(row['path'])
        if img_bgr is None:
            continue
        img_bgr   = cv2.resize(img_bgr, (32, 32))
        img_clahe = apply_clahe(img_bgr)

        rms_before = rms_contrast(cv2.cvtColor(img_bgr,   cv2.COLOR_BGR2GRAY))
        rms_after  = rms_contrast(cv2.cvtColor(img_clahe, cv2.COLOR_BGR2GRAY))
        improvement = rms_after - rms_before
        rms_improvements.append(improvement)

        ax_b, ax_a = (axes[i, 0], axes[i, 1]) if n_show > 1 else (axes[0], axes[1])
        ax_b.imshow(cv2.cvtColor(img_bgr,   cv2.COLOR_BGR2RGB))
        ax_b.set_title(f'Before  RMS={rms_before:.2f}', fontsize=9)
        ax_b.axis('off')

        ax_a.imshow(cv2.cvtColor(img_clahe, cv2.COLOR_BGR2RGB))
        ax_a.set_title(f'After CLAHE  RMS={rms_after:.2f}  (+{improvement:.2f})', fontsize=9)
        ax_a.axis('off')
    except Exception:
        pass

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'clahe_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

avg_improvement = np.mean(rms_improvements) if rms_improvements else 0
print(f"\nAverage RMS contrast improvement from CLAHE: {avg_improvement:.4f}")

Brightness 10th percentile threshold: 0.4120
Darkest images selected for CLAHE demo: 8



Average RMS contrast improvement from CLAHE: 19.0782


---
##  STEP 10 — Augmentation Preview (NO images saved)

In [13]:
# DATA SOURCE: training split only.
# NOTE: Images are ONLY displayed — NEVER saved to disk.

if T is None:
    print("torchvision not available. Skipping Step 10.")
else:
    augmentations = [
        ('Original',        T.Compose([])),
        ('HFlip',           T.RandomHorizontalFlip(p=1.0)),
        ('VFlip',           T.RandomVerticalFlip(p=1.0)),
        ('Rotate15',        T.RandomRotation(degrees=15)),
        ('Affine',          T.RandomAffine(degrees=10, translate=(0.1, 0.1), scale=(0.9, 1.1))),
        ('ColorJitter',     T.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.3, hue=0.1)),
        ('GaussianBlur',    T.GaussianBlur(kernel_size=5, sigma=(0.1, 2.0))),
        ('Perspective',     T.RandomPerspective(distortion_scale=0.4, p=1.0)),
    ]

    N_AUG = len(augmentations)  # 8 variations

    for label in labels_ordered:
        cls_df = train_df[train_df['label'] == label]
        sample_row = cls_df.sample(1, random_state=SEED).iloc[0]

        try:
            base_img = Image.open(sample_row['path']).convert('RGB').resize((224, 224))
        except Exception:
            print(f"Could not open image for {label}")
            continue

        fig, axes = plt.subplots(2, 4, figsize=(16, 8))
        fig.suptitle(f'Augmentation Preview — {label}', fontsize=13, fontweight='bold')

        for ax, (aug_name, transform) in zip(axes.flat, augmentations):
            aug_img = transform(base_img)
            ax.imshow(aug_img)
            ax.set_title(aug_name, fontsize=9)
            ax.axis('off')

        safe = label.replace(' ', '_').replace('-','').replace('/','_')
        plt.tight_layout()
        plt.savefig(OUTPUT_DIR / f'augmentation_preview_{safe}.png', dpi=120, bbox_inches='tight')
        plt.show()
        plt.close()

---
##  STEP 11 — Final Summary & Assertions

In [14]:
# DATA SOURCE: training split only (for all metrics below).

print("=" * 60)
print("    CottonGuard AI — EDA Final Summary")
print("=" * 60)

print(f"\n Dataset Overview")
print(f"  Total images (all)    : {len(df_all)}")
print(f"  Raw images            : {(~df_all['is_augmented']).sum()}")
print(f"  Augmented images      : {df_all['is_augmented'].sum()}")
print(f"  Corrupt files         : {len(corrupt_files)}")
print(f"  Non-image files       : {len(non_image_files)}")

print(f"\n  Split Sizes")
print(f"  Train (total)         : {len(train_df)}")
print(f"    ├= Real             : {len(df_train_real)}")
print(f"    └= Augmented        : {len(df_aug_copy)}")
print(f"  Validation            : {len(val_df)}")
print(f"  Test                  : {len(test_df)}")

print(f"\n Class Distribution (Training Set)")
for lbl, cnt in train_df['label'].value_counts().items():
    pct = cnt / len(train_df) * 100
    print(f"  {lbl:<38}: {cnt:>5}  ({pct:.1f}%)")

print(f"\n  Imbalance Ratio (max/min): {imbalance_ratio:.2f}")

print(f"\n Flagged Images (Training Set)")
print(f"  Unusual aspect ratio  : {len(flag_ar)}")
print(f"  Min dimension < 100px : {len(flag_sz)}")

print(f"\n Normalization Stats (training only)")
print(f"  Mean RGB              : {[round(v,4) for v in norm_stats['mean_rgb']]}")
print(f"  Std  RGB              : {[round(v,4) for v in norm_stats['std_rgb']]}")

print(f"\n CLAHE RMS Contrast Improvement")
print(f"  Average improvement   : {avg_improvement:.4f}")

print(f"\n GLCM ANOVA p-values (inter-class separability)")
for prop, res in anova_results.items():
    sig = " significant" if res['p_value'] < 0.05 else " not significant"
    print(f"  {prop:<15}: p={res['p_value']:.4e}  {sig}")

print(f"\n Saved Outputs")
for f in sorted(OUTPUT_DIR.iterdir()):
    print(f"  {f.name}")

print("\n" + "=" * 60)
print("    Data Leakage Assertions")
print("=" * 60)

# Assertion 1: No augmented images in val/test
assert val_df['is_augmented'].sum()  == 0, "LEAK: augmented images in validation!"
assert test_df['is_augmented'].sum() == 0, "LEAK: augmented images in test!"
print("   No augmented images in val/test.")

# Assertion 2: No overlap between train-real and val/test
train_real_paths = set(df_train_real['path'])
val_p  = set(val_df['path'])
test_p = set(test_df['path'])
assert len(train_real_paths & val_p)  == 0, "LEAK: train ∩ val!"
assert len(train_real_paths & test_p) == 0, "LEAK: train ∩ test!"
assert len(val_p & test_p)            == 0, "LEAK: val ∩ test!"
print("   No path overlap between splits.")

# Assertion 3: Normalization from training only
assert norm_stats['computed_from'] == 'training_split_only', "LEAK: norm stats not from training only!"
print("   Normalization stats computed from training split only.")

print("\n EDA Complete — All checks passed!")

   🌿 CottonGuard AI — EDA Final Summary

📦 Dataset Overview
  Total images (all)    : 3466
  Raw images            : 1178
  Augmented images      : 2288
  Corrupt files         : 0
  Non-image files       : 0

✂️  Split Sizes
  Train (total)         : 3112
    ├─ Real             : 824
    └─ Augmented        : 2288
  Validation            : 176
  Test                  : 177

📊 Class Distribution (Training Set)
  Alternaria Leaf                       :   690  (22.2%)
  Bacterial Blight - Mild               :   335  (10.8%)
  Bacterial Blight - Moderate           :   335  (10.8%)
  Bacterial Blight - Critical           :   313  (10.1%)
  Fussarium Wilt - Mild                 :   313  (10.1%)
  Fussarium Wilt - Moderate             :   312  (10.0%)
  Curl Virus - Mild                     :   306  (9.8%)
  Curl Virus - Moderate                 :   304  (9.8%)
  Curl Virus - Critical                 :   108  (3.5%)
  Fussarium Wilt - Critical             :    96  (3.1%)

⚖️  Imbalance Rati

## Phase 2.5: The PyTorch Bridge (Dataset & DataLoaders)

In [15]:
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
from PIL import Image

# 1. Define Label Mappings based on Phase 2
all_labels = sorted(train_df['label'].unique())
CLASS_TO_IDX = {label: idx for idx, label in enumerate(all_labels)}
IDX_TO_CLASS = {idx: label for label, idx in CLASS_TO_IDX.items()}

# 2. Convert Dataframes to Record Lists
train_records = train_df.to_dict('records')
val_records = val_df.to_dict('records')
test_records = test_df.to_dict('records')

# 3. Custom PyTorch Dataset
class CottonDataset(Dataset):
    def __init__(self, records, class_map, transform=None):
        self.records = records
        self.class_map = class_map
        self.transform = transform
        
    def __len__(self):
        return len(self.records)
        
    def __getitem__(self, idx):
        row = self.records[idx]
        img_path = row['path']
        img = Image.open(img_path).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, self.class_map[row['label']]

_mean = [0.485, 0.456, 0.406]
_std = [0.229, 0.224, 0.225]

train_transform = T.Compose([
    T.Resize((224, 224)),
    T.RandomHorizontalFlip(),
    T.ToTensor(),
    T.Normalize(mean=_mean, std=_std)
])
val_transform = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize(mean=_mean, std=_std)
])

train_dataset = CottonDataset(train_records, CLASS_TO_IDX, transform=train_transform)
val_dataset = CottonDataset(val_records, CLASS_TO_IDX, transform=val_transform)
test_dataset = CottonDataset(test_records, CLASS_TO_IDX, transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=0, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=0)

print(f"DataLoaders Ready: Train={len(train_loader)} batches, Val={len(val_loader)} batches")


DataLoaders Ready: Train=97 batches, Val=6 batches
